In [1]:
import pandas as pd
import sys

sys.path.insert(0, '..')
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

Challenge 2.1 — Explode + Merge + GroupBy

Question: How many errors does each severity level have,

and what's the most common error code within each severity?

Steps to think about:


You need individual error codes (not the comma string)

You need severity per error code (from the lookup table, not the transaction)

You need to group and count

In [3]:
# First, let's build an error code lookup table from validation_rules
from config.validation_rules import ALL_ERROR_CODES

error_lookup = pd.DataFrame([
    {'error_code':code, 'severity': info ['severity']}
    for code, info in ALL_ERROR_CODES.items()
])
print(f"Error lookup: {len(error_lookup)} codes")
error_lookup.head(60)

# Count codes per severity + list them
summary = (
  error_lookup
  .groupby('severity')
  .agg(
      count=('error_code', 'count'),
      codes=('error_code', lambda x: ', '.join(sorted(x)))
  )
  .reset_index()
)

print(summary.to_string(index=False))


Error lookup: 54 codes
severity  count                                                                                                                                                                                  codes
CRITICAL     15                                                                      AMT001, AMT002, AMT005, DATE004, DATE005, LC001, LC002, LC003, LC004, LC005, LC006, LC007, LC008, PTY001, XVAL004
    HIGH     22 AMT003, AMT004, BIC001, BIC002, BIC003, DATE001, DATE002, DATE008, LEI001, LEI002, LEI003, LEI004, PTY002, PTY004, PTY005, PTY006, PTY010, SHIP001, SHIP002, SHIP003, SHIP005, XVAL001
     LOW      2                                                                                                                                                                        LEI006, SHIP006
  MEDIUM     15                                                            AMT006, BIC004, DATE003, DATE006, DATE007, LEI005, PTY003, PTY007, PTY008, PTY009, PTY011, PTY012, PTY013,